# 01 — Real German addresses (OpenStreetMap)

Downloads OpenStreetMap extracts from Geofabrik for the regions in `00_config`, pulls out every address point
(street, house number, postcode, city, coordinates), removes duplicates and samples exactly
`N_USERS + N_HANDYMEN` **unique, real** addresses into `synth_address_pool`.
Each address is tagged `purpose = 'user' | 'handyman'`, so no user and handyman share an address.

Data © OpenStreetMap contributors (ODbL). Needs outbound internet access to `download.geofabrik.de`.

In [10]:
%pip install osmium requests

Note: you may need to restart the kernel to use updated packages.


In [11]:
try:
    dbutils.library.restartPython()
except NameError:  # running locally: no dbutils, nothing to restart
    pass

In [12]:
# Environment bootstrap: runs unchanged on Databricks and on a local machine.
# Locally it connects through Databricks Connect (serverless unless DATABRICKS_CLUSTER_ID is set)
# and provides `spark`, `display` and `list_volume_files`. Override the profile with DATABRICKS_CONFIG_PROFILE.
import os
IS_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ
LOCAL_SPARK_DEPENDENCIES = []   # pip packages the remote workers need (serverless Databricks Connect only)
if not IS_DATABRICKS:
    os.environ.setdefault("DATABRICKS_CONFIG_PROFILE", "george.sokolovsky2001@gmail.com")
    from databricks.connect import DatabricksSession

    _builder = DatabricksSession.builder
    if os.environ.get("DATABRICKS_CLUSTER_ID"):
        _builder = _builder.clusterId(os.environ["DATABRICKS_CLUSTER_ID"])
    else:
        _builder = _builder.serverless()
        if LOCAL_SPARK_DEPENDENCIES:
            from databricks.connect import DatabricksEnv
            _builder = _builder.withEnvironment(DatabricksEnv().withDependencies(LOCAL_SPARK_DEPENDENCIES))
    spark = _builder.getOrCreate()

    def display(obj, n=1000):
        from IPython.display import display as _display
        _display(obj.limit(n).toPandas() if hasattr(obj, "toPandas") else obj)

def list_volume_files(directory, suffix=""):
    """File names in a Unity Catalog Volume folder (the Volume is only mounted on Databricks, so go through the Files API locally)."""
    if IS_DATABRICKS:
        return [f for f in os.listdir(directory) if f.endswith(suffix)] if os.path.isdir(directory) else []
    from databricks.sdk import WorkspaceClient
    return [e.name for e in WorkspaceClient().files.list_directory_contents(directory) if e.name.endswith(suffix)]


In [2]:
%run ./00_config

In [14]:
# Cell 1 — Download the regional extracts to local disk
import os, requests, time

def _writable_dir(*candidates):
    """First candidate folder we can create and write to (serverless compute denies /local_disk0)."""
    for d in candidates:
        try:
            os.makedirs(d, exist_ok=True)
            with open(os.path.join(d, ".write_test"), "w"):
                pass
            return d
        except OSError:
            continue
    raise RuntimeError(f"No writable folder among {candidates}")

import tempfile
LOCAL_DIR = _writable_dir("/local_disk0/osm", os.path.join(tempfile.gettempdir(), "osm"))
os.makedirs(LOCAL_DIR, exist_ok=True)

def download(url, dest, retries=3):
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        return dest
    for attempt in range(1, retries + 1):
        try:
            with requests.get(url, stream=True, timeout=120, headers={"User-Agent": "maintops-capstone/1.0"}) as resp:
                resp.raise_for_status()
                with open(dest, "wb") as f:
                    for chunk in resp.iter_content(1 << 20):
                        f.write(chunk)
            return dest
        except Exception as e:
            print(f"  attempt {attempt} failed: {e}")
            time.sleep(5 * attempt)
    raise RuntimeError(f"Could not download {url}")

pbf_files = {}
for region, cfg in REGIONS.items():
    dest = os.path.join(LOCAL_DIR, f"{region.lower()}.osm.pbf")
    print(f"Downloading {region} ...")
    pbf_files[region] = download(cfg["url"], dest)
    print(f"  {os.path.getsize(dest) / 1e6:.0f} MB")

  99 MB
  54 MB
  21 MB
  55 MB


In [15]:
# Cell 2 — Extract address points (nodes with addr:* tags and buildings/ways with addr:* tags -> centroid)
import osmium
import pandas as pd

def extract_addresses(path, region):
    rows = []
    # with_locations() caches node coordinates so way centroids can be computed; the key filter keeps only tagged address objects
    fp = (osmium.FileProcessor(path, osmium.osm.NODE | osmium.osm.WAY)
          .with_locations()
          .with_filter(osmium.filter.KeyFilter("addr:housenumber")))
    for o in fp:
        tags = o.tags
        street, number, postcode, city = (tags.get("addr:street"), tags.get("addr:housenumber"),
                                          tags.get("addr:postcode"), tags.get("addr:city"))
        if not (street and number and postcode and city) or len(number) > 16 or len(street) > 100 or len(city) > 60:
            continue
        if o.is_node():
            if not o.location.valid():
                continue
            lat, lon = o.location.lat, o.location.lon
        else:
            pts = [(n.location.lat, n.location.lon) for n in o.nodes if n.location.valid()]
            if not pts:
                continue
            lat = sum(p[0] for p in pts) / len(pts)
            lon = sum(p[1] for p in pts) / len(pts)
        rows.append((region, street.strip(), number.strip(), postcode.strip(), city.strip(), round(lat, 6), round(lon, 6)))
    return pd.DataFrame(rows, columns=["state", "street", "housenumber", "postcode", "city", "latitude", "longitude"])

frames = {}
for region, path in pbf_files.items():
    print(f"Extracting {region} ...")
    df = extract_addresses(path, region)
    df = df.drop_duplicates(subset=["street", "housenumber", "postcode", "city"])
    frames[region] = df
    print(f"  {len(df):,} unique addresses")

Extracting Berlin ...
  407,255 unique addresses
Extracting Hamburg ...
  248,691 unique addresses
Extracting Bremen ...
  124,276 unique addresses
Extracting Saarland ...
  329,855 unique addresses


In [16]:
# Cell 3 — Sample the unique addresses per region and label them user / handyman
import numpy as np

rng = np.random.default_rng(SEED)
users_alloc = allocate(N_USERS)
handy_alloc = allocate(N_HANDYMEN)

parts = []
for region, df in frames.items():
    need = users_alloc[region] + handy_alloc[region]
    if len(df) < need:
        raise ValueError(f"{region}: only {len(df):,} unique addresses but {need:,} needed — lower its weight or add regions")
    sample = df.sample(n=need, random_state=SEED).reset_index(drop=True)
    sample["purpose"] = ["user"] * users_alloc[region] + ["handyman"] * handy_alloc[region]
    parts.append(sample)

pool = pd.concat(parts, ignore_index=True)
pool["country"] = "Germany"
pool = pool.sample(frac=1, random_state=SEED).reset_index(drop=True)        # shuffle across regions
pool.insert(0, "address_id", np.arange(1, len(pool) + 1, dtype="int64"))

assert not pool.duplicated(subset=["street", "housenumber", "postcode", "city"]).any()
print(pool.groupby(["state", "purpose"]).size())

state     purpose 
Berlin    handyman     4000
          user        40000
Bremen    handyman     1000
          user        10000
Hamburg   handyman     3000
          user        30000
Saarland  handyman     2000
          user        20000
dtype: int64


In [17]:
# Cell 4 — Save to Delta
(spark.createDataFrame(pool)
      .write.mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable(T_ADDRESS_POOL))
print(f"Saved {len(pool):,} addresses to {T_ADDRESS_POOL}")
display(spark.table(T_ADDRESS_POOL).limit(10))

Saved 110,000 addresses to bootcamp_students.maintops.synth_address_pool


,address_id,state,street,housenumber,postcode,city,latitude,longitude,purpose,country
0,27501,Berlin,Luxemburger Straße,12,13353,Berlin,52.543904,13.353528,user,Germany
1,27502,Hamburg,Baumkamp,33,22299,Hamburg,53.599472,9.999406,user,Germany
2,27503,Bremen,Husumer Straße,6,28219,Bremen,53.101382,8.790745,handyman,Germany
3,27504,Bremen,Querlandstraße,8f,28357,Bremen,53.122813,8.901496,user,Germany
4,27505,Bremen,H.-H.-Meier-Allee,19,28213,Bremen,53.090412,8.845189,user,Germany
5,27506,Hamburg,Diestelstraße,30b,22397,Hamburg,53.689940,10.126930,user,Germany
6,27507,Berlin,Weislingenstraße,21,13465,Berlin,52.641298,13.302396,user,Germany
7,27508,Saarland,Mecklenburgring,24,66121,Saarbrücken,49.237995,7.039115,user,Germany
8,27509,Berlin,Ruwersteig,32,12681,Berlin,52.521631,13.533750,user,Germany
9,27510,Bremen,Louis-Seegelken-Straße,32a,28717,Bremen,53.180251,8.690128,handyman,Germany
